# Instructor Notebook 04 — Transformers & Attention
**ComplianceGPT Lab · REU 2026 · Week 2**

> **Teaching script:** Run every cell top-to-bottom. No GPU required. No HuggingFace model downloads.

**Learning arc:**
1. Tokenization — how text becomes integers
2. The RNN problem — why sequential processing fails
3. Self-attention — every token attends to every token
4. Context-dependent meaning: 'order' in medical vs legal context
5. Positional encoding — how the model knows word order
6. Why this matters for HIPAA: long-range dependencies

In [ ]:
import numpy as np
import pandas as pd
import json
import warnings
warnings.filterwarnings('ignore')

# Try to load tiktoken (fastest tokenizer demo)
try:
    import tiktoken
    enc = tiktoken.encoding_for_model('gpt-4')
    TOKENIZER = 'tiktoken (GPT-4 vocab, 100k tokens)'
    def tokenize(text):
        return enc.encode(text)
    def decode_token(tok_id):
        return enc.decode([tok_id])
    HAS_TIKTOKEN = True
except ImportError:
    HAS_TIKTOKEN = False
    TOKENIZER = 'simple whitespace tokenizer (tiktoken not installed)'
    vocab = {}
    def tokenize(text):
        words = text.lower().split()
        return [vocab.setdefault(w, len(vocab)) for w in words]
    def decode_token(tok_id):
        rev = {v:k for k,v in vocab.items()}
        return rev.get(tok_id, f'<{tok_id}>')

print(f'Tokenizer: {TOKENIZER}')

---
## Part 1 · Tokenization

> **Say:** "Before anything else, LLMs convert text into integers. Not words — *subword pieces*. The vocabulary has ~100,000 tokens. Common words get their own token. Rare words get split into pieces."

> **Say:** "This matters for HIPAA: legal jargon like 'subpoena' might tokenize differently than 'sub-poe-na'. The model sees the pieces, not the whole word."

In [ ]:
demo_texts = [
    "The court issued a subpoena for medical records.",
    "Patient authorization is required before disclosure.",
    "HIPAA §164.512(e) permits disclosure with a court order.",
    "The hospital violated 45 CFR Part 164 requirements.",
]

print('Tokenization demo:')
print('='*70)
for text in demo_texts:
    tokens = tokenize(text)
    print(f'\nText: {text}')
    print(f'Token count: {len(tokens)}')
    if HAS_TIKTOKEN:
        pieces = [repr(decode_token(t)) for t in tokens]
        print(f'Pieces: {" | ".join(pieces)}')
    else:
        print(f'Token IDs: {tokens}')

In [ ]:
# Load HIPAA data and count tokens
HIPAA_PATH = '/Users/priscilladanso/Documents/GitHub/COMPLIANCEGPT/experiments/finalserverrun/final_vast_gemma3_4b.csv'
hipaa = pd.read_csv(HIPAA_PATH)
texts = hipaa['question'].fillna('').tolist()

# Token counts per scenario
token_counts = [len(tokenize(t)) for t in texts]
tc = pd.Series(token_counts)

print('Token counts across 137 HIPAA scenarios:')
print(f'  Minimum:   {tc.min()} tokens')
print(f'  Median:    {tc.median():.0f} tokens')
print(f'  Maximum:   {tc.max()} tokens')
print(f'  Mean:      {tc.mean():.0f} tokens')
print()
print('Context window of Gemma3-4B: 8,192 tokens')
print(f'Longest scenario uses {tc.max()/8192:.1%} of the context window')
print(f'Average scenario uses {tc.mean()/8192:.1%} of the context window')
print()
print('Histogram of token counts:')
bins = [0, 100, 200, 300, 400, 500, 600, 1000]
for i in range(len(bins)-1):
    count = ((tc >= bins[i]) & (tc < bins[i+1])).sum()
    bar = '█' * count
    print(f'  {bins[i]:>4}–{bins[i+1]:<4} tokens: {bar} ({count})')

---
## Part 2 · The RNN Problem — Why Sequential Fails

> **Say:** "Before Transformers, NLP used Recurrent Neural Networks. RNNs process text one token at a time, left to right. Each step passes a 'hidden state' to the next step. It's like reading a sentence one word at a time and trying to remember everything."

> **Say:** "The problem: information from early in the sequence gets compressed and diluted as you pass through many steps. By the time you read token 100, token 1 has faded."

> **This is the vanishing gradient problem.**

In [ ]:
# Simulate information decay in an RNN
# Each step: hidden_state = 0.9 * hidden_state + 0.1 * new_input
# Signal from step 0 fades exponentially

def rnn_memory_retention(n_steps, decay=0.9):
    """How much of the first token's signal remains after n_steps."""
    signal = 1.0
    for _ in range(n_steps):
        signal *= decay
    return signal

print('RNN information retention: how much of token-1 survives at token N?')
print()
print(f"  {'After N steps':>20} {'Signal remaining':>18} {'Visual'}")
print('  ' + '-' * 60)

hipaa_example = (
    "The patient, who was admitted after a workplace injury treated "
    "by a physician who had previously disclosed records in an "
    "unrelated matter, filed for damages claiming the hospital had "
    "shared information without court authorization."
)

checkpoints = [1, 5, 10, 20, 50, len(hipaa_example.split())]
for n in checkpoints:
    retain = rnn_memory_retention(n)
    bar = '█' * int(retain * 30)
    fade = '░' * (30 - int(retain * 30))
    print(f'  Token {n:>4}          {retain:>17.3%}  [{bar}{fade}]')

print()
print('Example long sentence:')
print(f'  "{hipaa_example[:80]}..."')
print(f'  Subject ("patient") is at token 2.')
print(f'  Key fact ("without court authorization") is at token {len(hipaa_example.split())}.')
print(f'  Signal from "patient" reaching the end: {rnn_memory_retention(len(hipaa_example.split())):.3%}')
print(f'  → RNN has essentially forgotten who the subject is by the time it reads the key fact.')

---
## Part 3 · Self-Attention — Every Token to Every Token

> **Say:** "The Transformer's solution: throw out sequential processing entirely. Every token attends to every other token simultaneously. No distance penalty. Token 1 can directly influence token 100."

> **Say:** "Attention works like this: for each token, compute a score for how much it should 'look at' every other token. High score = high influence on this token's new representation."

In [ ]:
# Manual attention demonstration
# We'll show how 'order' in two different sentences attends differently

def softmax(x):
    e = np.exp(x - np.max(x))
    return e / e.sum()

# Sentence: "The judge signed the order to release records"
# We simulate what the word 'order' attends to
sentence_legal = ["The", "judge", "signed", "the", "order", "to", "release", "records"]

# These are SIMULATED attention scores (in real transformers, learned from data)
# Showing: 'order' (position 4) attends to relevant context tokens
attention_legal = np.array([0.05, 0.35, 0.25, 0.02, 1.0, 0.05, 0.15, 0.20])
weights_legal = softmax(attention_legal)

print('Self-Attention: what does "order" attend to in a LEGAL context?')
print('Sentence: "The judge signed the order to release records"')
print()
print(f"  {'Token':<12} {'Raw score':>10} {'Attention weight':>16}  Visual")
print('  ' + '-' * 60)
for i, (tok, raw, w) in enumerate(zip(sentence_legal, attention_legal, weights_legal)):
    bar = '█' * int(w * 50)
    marker = ' ← THIS WORD' if i == 4 else ''
    print(f"  {tok:<12} {raw:>10.2f} {w:>16.3f}  {bar}{marker}")

print()
print('→ "order" attends strongly to "judge" and "signed" — legal action context')
print('  → Model will interpret "order" as COURT ORDER → has_court_order = True')

In [ ]:
# Same word 'order' in a MEDICAL context
sentence_medical = ["The", "physician", "wrote", "a", "medication", "order", "for", "antibiotics"]

# 'order' (position 5) now attends to different tokens
attention_medical = np.array([0.05, 0.30, 0.15, 0.02, 0.40, 1.0, 0.10, 0.25])
weights_medical = softmax(attention_medical)

print('Self-Attention: what does "order" attend to in a MEDICAL context?')
print('Sentence: "The physician wrote a medication order for antibiotics"')
print()
print(f"  {'Token':<12} {'Raw score':>10} {'Attention weight':>16}  Visual")
print('  ' + '-' * 60)
for i, (tok, raw, w) in enumerate(zip(sentence_medical, attention_medical, weights_medical)):
    bar = '█' * int(w * 50)
    marker = ' ← THIS WORD' if i == 5 else ''
    print(f"  {tok:<12} {raw:>10.2f} {w:>16.3f}  {bar}{marker}")

print()
print('→ "order" now attends to "physician" and "medication" — medical context')
print('  → Model interprets "order" as MEDICAL ORDER → has_court_order = False')
print()
print('='*65)
print('SAME WORD. DIFFERENT CONTEXT. DIFFERENT MEANING.')
print('This is what TF-IDF cannot do. This is what transformers do.')
print('='*65)

---
## Part 4 · Attention Math (Optional Deep Dive)

> **Say:** "If you want to understand exactly how attention weights are computed, here's the math. This is directly from 'Attention is All You Need' (2017)."

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

- **Q** (Query): What is this token looking for?
- **K** (Key): What does each token offer?
- **V** (Value): What information does each token carry?
- **√dₖ**: Scaling factor to prevent very large dot products

In [ ]:
np.random.seed(42)

# Simulate scaled dot-product attention with small d_model
d_model = 4   # embedding dimension (real: 4096)
seq_len = 5   # sequence length

# Learned weight matrices (in real model, these are trained)
W_Q = np.random.randn(d_model, d_model) * 0.5
W_K = np.random.randn(d_model, d_model) * 0.5
W_V = np.random.randn(d_model, d_model) * 0.5

# Input embeddings (5 tokens, 4 dims each)
X = np.random.randn(seq_len, d_model)

# Compute Q, K, V
Q = X @ W_Q
K = X @ W_K
V = X @ W_V

# Scaled dot-product attention
scores = Q @ K.T / np.sqrt(d_model)
attn_weights = np.array([softmax(scores[i]) for i in range(seq_len)])
output = attn_weights @ V

print(f'Self-Attention computation (toy: {seq_len} tokens, d_model={d_model})')
print()
print('Attention weight matrix (how much each token attends to each other):')
print('  Rows = query token, Cols = key token (what it attends to)')
print()
print(f"  {'':>6}" + ''.join(f'  tok{j}' for j in range(seq_len)))
print('  ' + '-' * (6 + 6*seq_len))
for i in range(seq_len):
    vals = ''.join(f'  {attn_weights[i][j]:.2f}' for j in range(seq_len))
    print(f'  tok{i}: {vals}')

print()
print('Output shape:', output.shape, '(each token now has context from all other tokens)')
print()
print('In Gemma3-4B: d_model=2048, seq_len up to 8192')
print('Attention matrix: 8192 × 8192 = 67M values — computed in parallel on GPU')

---
## Part 5 · Positional Encoding — How Order Is Preserved

> **Say:** "Wait — if every token attends to every other token simultaneously, how does the model know word order? 'Court order' and 'order court' look the same to a permutation-invariant attention layer."

> **Say:** "Transformers add a positional encoding to each token embedding. This is a vector that encodes the position. The model learns to read position from the combined embedding."

In [ ]:
def positional_encoding(seq_len, d_model):
    """Sinusoidal positional encoding from 'Attention is All You Need'."""
    PE = np.zeros((seq_len, d_model))
    positions = np.arange(seq_len)[:, np.newaxis]
    dims = np.arange(0, d_model, 2)
    PE[:, 0::2] = np.sin(positions / (10000 ** (dims / d_model)))
    PE[:, 1::2] = np.cos(positions / (10000 ** (dims[:d_model//2] / d_model)))
    return PE

PE = positional_encoding(seq_len=10, d_model=8)

print('Positional Encoding (10 positions, 8 dimensions):')
print('Each row is the encoding added to the token embedding at that position.')
print()
print(f"  {'Pos':>4}  " + '  '.join(f'd{i}' for i in range(8)))
print('  ' + '-' * 60)
for i, row in enumerate(PE):
    vals = '  '.join(f'{v:+.2f}' for v in row)
    print(f'  {i:>4}  {vals}')

print()
print('Key property: each position has a unique encoding.')
print('Nearby positions have similar encodings (smooth continuity).')
print('Model can compute relative positions from these values.')

---
## Part 6 · Why This Matters for HIPAA

> **Say:** "Let's look at a real HIPAA scenario that requires long-range reasoning. This is the kind of thing RNNs fail on and Transformers handle."

In [ ]:
# Find a long HIPAA scenario that requires connecting distant facts
long_idx = pd.Series([len(t.split()) for t in texts]).nlargest(5).index.tolist()

for idx in long_idx[:2]:
    text = texts[idx]
    verdict = hipaa['ground_truth'].iloc[idx]
    pred    = hipaa['verdict_norm'].iloc[idx]
    matched = hipaa['match'].iloc[idx]
    words   = text.split()
    
    print(f'Case {hipaa["row_id"].iloc[idx]} — {len(words)} words')
    print(f'  Ground truth: {verdict}  |  LLM prediction: {pred}  |  Match: {matched}')
    print()
    print('  First 100 words (WHO and WHAT):') 
    print(' ', ' '.join(words[:100]))
    print()
    print('  Last 50 words (KEY FACT):') 
    print(' ', ' '.join(words[-50:]))
    print()
    print('  → The model must connect the subject (beginning) to the key legal fact (end).')
    print('  → This requires attending across 200+ token distance.')
    print('  → RNNs struggle here. Transformers handle it directly.')
    print('=' * 70)
    print()

---
## Part 7 · Transformer Architecture Overview

> **Say:** "Let's put it all together: what does one Transformer layer actually do to a sequence?"

In [ ]:
# Print transformer architecture walkthrough
steps = [
    ('INPUT',          'Raw text: "The judge signed a court order"'),
    ('TOKENIZE',       '→ [512, 4008, 9431, 287, 2267, 1502]  (integer IDs)'),
    ('EMBED',          '→ Each ID → 4096-dim vector  (lookup table, trained)'),
    ('ADD POSITION',   '→ Add positional encoding to each embedding'),
    ('LAYER 1',        '→ Multi-head self-attention: every token attends to all others'),
    ('             ',  '   Each token updates its representation based on attention weights'),
    ('             ',  '   + Feed-forward network (2 dense layers)  + residual connection'),
    ('LAYER 2–32',    '→ Same as Layer 1, but learns different attention patterns'),
    ('             ',  '   Early layers: syntax  |  Mid layers: semantics  |  Late: reasoning'),
    ('OUTPUT',         '→ Final hidden states → vocabulary scores → next token probabilities'),
    ('DECODE',         '→ Pick highest probability token → append → repeat'),
]

print('Transformer forward pass — step by step:')
print()
for step, desc in steps:
    print(f'  {step:<15} {desc}')

print()
print('Gemma3-4B specifications:')
specs = [
    ('Parameters',        '4 billion (4,000,000,000)'),
    ('Layers',            '34 transformer layers'),
    ('d_model',           '2048 (embedding dimension)'),
    ('Attention heads',   '8 (each head learns different patterns)'),
    ('Context window',    '8,192 tokens'),
    ('Vocabulary',        '256,128 tokens'),
    ('GPU memory',        '~8 GB at 4-bit quantization (fits on V100)'),
]
for spec, val in specs:
    print(f'  {spec:<20} {val}')

---
## Summary

| Concept | What it means |
|---|---|
| **Tokenization** | Text → integer IDs (subword vocabulary, ~100k tokens) |
| **Embedding** | Integer ID → dense vector in high-dimensional space |
| **Self-Attention** | Every token attends to every token — O(n²) but parallelizable |
| **Context-dependence** | "order" = court order or medication order depending on context |
| **Positional encoding** | Preserves word order despite permutation-invariant attention |
| **Depth** | 32 layers refine representations from syntax → semantics → reasoning |

> **Transition:** "Transformers give us context-dependent representations. But there are two families: Encoders (BERT — reads whole sentence bidirectionally) and Decoders (GPT/Gemma — generates left to right). Next: we use an encoder model to get sentence embeddings and finally close the semantic gap."